<a href="https://colab.research.google.com/github/fjdvm/skin-disease-classifier/blob/main/skin_disease_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import pandas as pd
import numpy as np
from PIL import Image
from enum import unique
from sklearn.model_selection import train_test_split
from google.colab import userdata
import os

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:


token = userdata.get('KAGGLE_TOKEN')

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
  f.write(token)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

*Skin Cancer MNIST: HAM10000*

https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d ham10000_data
!ls ham10000_data

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
skin-cancer-mnist-ham10000.zip: Skipping, found more recently modified local copy (use --force to force download)
replace ham10000_data/HAM10000_images_part_1/ISIC_0024306.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
df = pd.read_csv("ham10000_data/HAM10000_metadata.csv")
print(df.shape)
df.head()

df['dx'].value_counts()

In [ ]:
df['lesion_id'].value_counts().head()

In [ ]:


# Get unique lesion_ids
unique_lesions = df['lesion_id'].unique()

# Split lesion IDs first (not rows)
train_lesions, temp_lesions = train_test_split(unique_lesions, test_size=0.3, random_state=42)
val_lesions, test_lesions = train_test_split(temp_lesions, test_size=0.5, random_state=42)

# Filter the dataframe based on which split each lesion belongs to
train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
val_df = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
test_df = df[df['lesion_id'].isin(test_lesions)].reset_index(drop=True)

print(f"Train: {len(train_df)} images, {len(train_lesions)} lesions")
print(f"Val: {len(val_df)} images, {len(val_lesions)} lesions")
print(f"Test: {len(test_df)} images, {len(test_lesions)} lesions")



In [ ]:
set(train_df['lesion_id']) & set(val_df['lesion_id'])

In [ ]:
import os

def get_image_path(img_id):
  path1 = f'ham10000_data/HAM10000_images_part1/{img_id}.jpg'
  path2 = f'ham10000_data/HAM10000_images_part_2/{img_id}.jpg'
  if os.path.exists(path1):
    return path1
  elif os.path.exists(path2):
    return path2
  else:
    raise FileNotFoundError(f"{img_id} not found in either folder")

  test_id = df.iloc[0]['img_id']
  print(get_image_path(test_id))

  classes = sorted(df['dx'].unique())
  class_to_idx = {cls: idx for ids, cls in enumerate(classes)}
  print(class_to_idx)

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms

class SkinLesionDataset(Dataset):
  def __init__(self, dataframe, class_to_idx, transform=None):
    self.df = dataframe
    self.class_to_idx = class_to_idx
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img_path = get_image_path(row['image_id'])
    image = Image.open(img_path).convert('RGB')
    label = self.class_to_idx[row['dx']]

    if self.transform:
      image = self.transform(image)

    return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SkinLesionDataset(train_df, class_to_idx, transform=train_transform)
val_dataset = SkinLesionDataset(val_df, class_to_idx, transform=val_test_transform)
test_dataset = SkinLesionDataset(test_df, class_to_idx, transofrm=val_test_transform)

# quick sanity check
img, label = train_datset[0]
print(img.shape, label)